In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
path = "online_retail_II.xlsx"

df_2009 = pd.read_excel(path, sheet_name='Year 2009-2010')
df_2010 = pd.read_excel(path, sheet_name='Year 2010-2011')
df = pd.concat([df_2009, df_2010], ignore_index=True)
df 

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [4]:
print(f"Check missing values: \n{df.isna().sum()}")

Check missing values: 
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


In [5]:
print(f"Check duplicated rows: \n{df.duplicated().sum()}")

Check duplicated rows: 
34335


In [6]:
import re
# Replace every digit with "9" and every letter with "A" so we can see
# the *shape* of a value (e.g. "85123A" -> "99999A") without caring
# about the specific digits/letters. Values with the same shape get
# grouped together, so we can spot which shapes are common/normal
# and which are rare/suspicious.
def number(s):
      s = re.sub(r'[0-9]', '9', str(s))
      #s = re.sub(r'[A-Za-z]', 'A', s)
      return s

def letter(s):
      s = re.sub(r'[A-Za-z]', 'A', s)
      return s 

# InvoiceNo: Invoice number. Nominal, a 6-digit integral number uniquely assigned to each transaction. 
# If this code starts with letter 'c', it indicates a cancellation. 
print(df['Invoice'].apply(number).value_counts())
non_c = df[df['Invoice'].astype(str).str.startswith('A')]
print(non_c)

Invoice
999999     1047871
C999999      19494
A999999          6
Name: count, dtype: int64
        Invoice StockCode      Description  Quantity         InvoiceDate  \
179403  A506401         B  Adjust bad debt         1 2010-04-29 13:36:00   
276274  A516228         B  Adjust bad debt         1 2010-07-19 11:24:00   
403472  A528059         B  Adjust bad debt         1 2010-10-20 12:04:00   
825443  A563185         B  Adjust bad debt         1 2011-08-12 14:50:00   
825444  A563186         B  Adjust bad debt         1 2011-08-12 14:51:00   
825445  A563187         B  Adjust bad debt         1 2011-08-12 14:52:00   

           Price  Customer ID         Country  
179403 -53594.36          NaN  United Kingdom  
276274 -44031.79          NaN  United Kingdom  
403472 -38925.87          NaN  United Kingdom  
825443  11062.06          NaN  United Kingdom  
825444 -11062.06          NaN  United Kingdom  
825445 -11062.06          NaN  United Kingdom  


In [7]:
df['StockCode'] = df['StockCode'].astype(str)
df['StockCode'] = df['StockCode'].str.strip()   

In [8]:
# StockCode: Product (item) code. Nominal, a 5-digit integral number uniquely assigned to each distinct product.
# Description: Product (item) name. Nominal.
print(df['StockCode'].apply(number).apply(letter).value_counts())


StockCode
99999           932385
99999A          127521
AAAA              2158
A                 1713
AAA               1446
99999AA           1372
A9                 283
AAAA9999           103
AAAA AAAAAAA       102
AAAA_9999_99       100
AAAAAAAAA           69
AAAAAA              67
AAAAAAAA            24
AAAA999             17
AAAA9999A            5
AAAAAA9              3
AA9999               3
Name: count, dtype: int64


In [9]:
df['Description'] = df['Description'].str.strip()
print(df['Description'].value_counts())

Description
WHITE HANGING HEART T-LIGHT HOLDER    5918
REGENCY CAKESTAND 3 TIER              4412
JUMBO BAG RED RETROSPOT               3469
ASSORTED COLOUR BIRD ORNAMENT         2958
PARTY BUNTING                         2765
                                      ... 
entry error                              1
Damages/samples                          1
MISELTOE HEART WREATH CREAM              1
HAYNES MINI-COOPER PLAYING CARDS         1
HAPPY BIRTHDAY GINGER CAT CARD           1
Name: count, Length: 5651, dtype: int64


In [10]:
# 72801C -> ['S/4 ROSE PINK DINNER CANDLES', '4 ROSE PINK DINNER CANDLES', 'check']
# 72801D -> ['S/4 SKY BLUE DINNER CANDLES', '4 SKY BLUE DINNER CANDLES']
# 72801c -> ['S/4 ROSE PINK DINNER CANDLES', '4 ROSE PINK DINNER CANDLES']
# 72801d -> ['S/4 SKY BLUE DINNER CANDLES', '4 SKY BLUE DINNER CANDLES']

df['StockCode'] = df['StockCode'].str.upper()

In [11]:
desc_variation = df.groupby('StockCode')['Description'].nunique()
print(desc_variation.value_counts().sort_index())
multi_desc = desc_variation[desc_variation > 1].index

for code in multi_desc:
    variants = df[df['StockCode'] == code]['Description'].unique()
    print(code, '->', list(variants))

Description
0     353
1    3587
2     926
3     193
4      55
5       9
6       3
7       4
9       1
Name: count, dtype: int64
10080 -> ['GROOVY CACTUS INFLATABLE', nan, 'check']
10120 -> ['DOGGY RUBBER', 'Zebra invcing error']
10133 -> ['COLOURING PENCILS BROWN TUBE', 'damaged']
15056N -> ['EDWARDIAN PARASOL NATURAL', 'wedding co returns?']
15058A -> ['BLUE WHITE SPOTS GARDEN PARASOL', 'BLUE POLKADOT GARDEN PARASOL', nan, 'wet/rusty']
15058B -> ['PINK WHITE SPOTS GARDEN PARASOL', 'PINK POLKADOT GARDEN PARASOL', nan]
15058C -> ['ICE CREAM DESIGN GARDEN PARASOL', 'wet/rusty']
16008 -> ['SMALL FOLDING SCISSOR(POINTED EDGE)', 'check']
16012 -> ['FOOD/DRINK SPUNGE STICKERS', 'FOOD/DRINK SPONGE STICKERS']
16045 -> ['POPART WOODEN PENCILS ASST', nan, 'check']
16151A -> ['FLOWER DES BLUE HANDBAG/ORANG HANDL', nan, 'found', 'FLOWERS HANDBAG blue and orange']
16156L -> ['WRAP, CAROUSEL', 'WRAP CAROUSEL']
16161U -> ['WRAP,SUKI AND FRIENDS', 'WRAP SUKI AND FRIENDS']
16162M -> [nan, 'THE KING GIF

In [12]:
def pick_description(variants):
    variants = variants.dropna()
    if variants.empty:
        return None
    upper_variants = variants[variants == variants.str.upper()]
    if not upper_variants.empty:
        return max(upper_variants, key=len)
    return max(variants, key=len)  # fallback if none are fully uppercase

stockcode_to_desc = df.groupby('StockCode')['Description'].agg(pick_description)

for code in desc_variation.sort_values(ascending=False).head(100).index:
    variants = df[df['StockCode'] == code]['Description']
    print(code, '->', pick_description(variants))

20713 -> JUMBO BAG OWLS
23084 -> RABBIT NIGHT LIGHT
22734 -> SET OF 6 RIBBONS VINTAGE CHRISTMAS
21181 -> PLEASE ONE PERSON  METAL SIGN
22423 -> REGENCY CAKESTAND 3 TIER
21830 -> MERCHANT CHANDLER CREDIT ERROR, STO
85175 -> CACTI T-LIGHT CANDLES
47566B -> TEA TIME PARTY BUNTING
85172 -> HYACINTH BULB T-LIGHT CANDLES
22501 -> PICNIC BASKET WICKER LARGE
21843 -> RED RETROSPOT CAKE STAND
72807A -> SET/3 ROSE CANDLE IN JEWELLED BOX
85123A -> WHITE HANGING HEART T-LIGHT HOLDER
22740 -> POLKA DOT PEN
22719 -> GUMBALL COATHOOK., BLACK & WHITE
23131 -> MISTLETOE HEART WREATH CREAM
22502 -> PICNIC BASKET WICKER 60 PIECES
22687 -> DOORMAT CHRISTMAS VILLAGE
84406B -> CREAM CUPID HEARTS COAT HANGER
85185B -> PINK HORSE SOCK PUPPET KIT
84016 -> FLAG OF ST GEORGE CAR FLAG
79000 -> MOROCCAN TEA GLASS
22346 -> PARTY PIZZA DISH GREEN+WHITE SPOT
21539 -> RED RETROSPOT BUTTER DISH
20665 -> RED RETROSPOT PURSE
23236 -> STORAGE TIN VINTAGE DOILEY
35965 -> FOLKART HEART NAPKIN RINGS
22121 -> NOEL WOODEN BLOC

In [16]:
df['Shape'] = df['StockCode'].apply(number).apply(letter)

correct_shapes = ['99999', '99999A', '99999AA']
minor_shapes = [s for s in df['Shape'].unique() if s not in correct_shapes ]
for shape in minor_shapes:
    print(shape)
    print(df[df['Shape'] == shape]['StockCode'].value_counts())
    print()

AAAA
StockCode
POST    2122
PADS      19
CRUK      16
GIFT       1
Name: count, dtype: int64

A
StockCode
M    1426
D     177
S     104
B       6
Name: count, dtype: int64

AAAA9999
StockCode
DCGS0058    31
DCGS0076    15
DCGS0003    14
DCGS0069     6
DCGS0004     5
DCGS0072     4
DCGS0070     3
DCGS0068     3
DCGS0062     2
DCGS0037     2
DCGS0074     1
DCGS0055     1
DCGS0041     1
DCGS0075     1
DCGS0073     1
DCGS0071     1
DCGS0057     1
DCGS0059     1
DCGS0053     1
DCGS0056     1
DCGS0060     1
DCGS0039     1
DCGS0036     1
DCGS0027     1
DCGS0016     1
DCGS0006     1
DCGS0044     1
DCGS0067     1
Name: count, dtype: int64

AAA
StockCode
DOT    1446
Name: count, dtype: int64

A9
StockCode
C2    282
C3      1
Name: count, dtype: int64

AAAA AAAAAAA
StockCode
BANK CHARGES    102
Name: count, dtype: int64

AAAA999
StockCode
TEST001    15
TEST002     2
Name: count, dtype: int64

AAAA_9999_99
StockCode
GIFT_0001_20    29
GIFT_0001_30    29
GIFT_0001_10    16
GIFT_0001_50     8
GIFT_0

In [ ]:
for code in ['C2', 'C3', 'GIFT']:
    print(code)
    print(df[df['StockCode'] == code]['Description'].value_counts())
    print()

C2
Description
CARRIAGE    279
Name: count, dtype: int64

C3
Series([], Name: count, dtype: int64)

GIFT
Series([], Name: count, dtype: int64)



In [18]:
admin_codes = ['POST', 'PADS', 'CRUK', 'DOT', 'BANK CHARGES', 'AMAZONFEE',
               'M', 'D', 'S', 'B', 'ADJUST', 'ADJUST2', 'TEST001', 'TEST002']

admin_rows = df[df['StockCode'].isin(admin_codes)]
print(admin_rows.groupby('StockCode')['Customer ID'].apply(lambda x: x.isna().sum()))
print()
print(admin_rows.groupby('StockCode')['Customer ID'].apply(lambda x: x.notna().sum()))

StockCode
ADJUST             6
ADJUST2            0
AMAZONFEE         43
B                  6
BANK CHARGES      64
CRUK               0
D                  3
DOT             1430
M                311
PADS               0
POST             103
S                104
TEST001            0
TEST002            1
Name: Customer ID, dtype: int64

StockCode
ADJUST            61
ADJUST2            3
AMAZONFEE          0
B                  0
BANK CHARGES      38
CRUK              16
D                174
DOT               16
M               1115
PADS              19
POST            2019
S                  0
TEST001           15
TEST002            1
Name: Customer ID, dtype: int64
